# Structured Output এবং Function Calling

**PART A — CONSTRAINED DECODING**, সত্যিই from scratch implement করা। আমরা একটা restricted JSON object-এর জন্য একটা ছোট্ট grammar define করি:

    {"action": "add" | "sub" | "mul", "value": <1-to-3-digit integer>}

আর একটা character-level state machine, `next_valid_chars(prefix)`, যা এ-পর্যন্ত-generate-করা string-টা inspect করে ঠিক সেই characters-গুলোর set return করে যেগুলো পরবর্তীতে emit করা legal (খালি set মানে structure ইতিমধ্যে সম্পূর্ণ এবং generation বন্ধ করতে হবে)। একটা প্রকৃত (কিন্তু উদ্দেশ্য নিয়ে UNTRAINED, random-ভাবে initialized) ছোট্ট decoder-only Transformer-এর প্রতিটি generation step-এ আমরা model-এর raw logits-গুলো character vocabulary-র ওপর নিই এবং valid set-এ না থাকা প্রতিটি character-এর logit-কে sampling-এর আগে -infinity করি — এটাই constrained decoding। তারপর অসংখ্য independent generation-এ দেখাই যে model-এর weights pure random noise হওয়া সত্ত্বেও আউটপুট **প্রতিবার** 100% syntactically valid, আর তার বিপরীতে দেখাই একই random model কোনো mask ছাড়া sample করলে প্রায় কখনোই valid structured output দেয় না। এখানে validity পুরোপুরি mask-থেকে আসে, model কিছু শেখা থেকে নয়।

**PART B — FUNCTION CALLING**, end to end scripted। একটা "model output" (একটা fixed string, যা বাস্তব LLM যেটা emit করত তার প্রতিনিধি) এখন-মান-সম্মত `{"name": ..., "arguments": {...}}` format-এ একটা structured function-call request ধারণ করে। বাইরের code সেই JSON parse করে, parsed arguments দিয়ে একটা REAL Python function-এ (calculator আর একটা ছোট্ট unit-conversion lookup) dispatch করে, execute করে, আর REAL return value-টা একটা final response-এ splice করে — যেটা প্রতিটি production tool-calling LLM API-র নিচে থাকে সেই পূর্ণ request -> parse -> execute -> respond loop-টা প্রদর্শন করে।

**Runtime:** CPU-তে কয়েক সেকেন্ড (একটা ছোট্ট untrained model character-level forward passes করছে, কোনো training loop-ই নেই)।

**চালানোর নিয়ম:**
- উপর থেকে নিচে cell-গুলো ক্রমান্বয়ে চালাও।
- আসল স্ক্রিপ্ট: `python example.py`

In [ ]:
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part A — Constrained decoding: guaranteed-valid structured output

Grammar-এর character-level state machine (`next_valid_chars`), একটা independent full-string validity checker, আর একটা genuinely UNTRAINED (random weights) Transformer define করা হয়েছে। Cell-এর শেষে `part_a_demo()` চলে।

In [ ]:
# ===========================================================================
# PART A: CONSTRAINED DECODING -- নিশ্চিত-valid STRUCTURED OUTPUT
# ===========================================================================

TEMPLATE_HEAD = '{"action": "'
TEMPLATE_MID = '", "value": '
ACTIONS = ["add", "sub", "mul"]

VALID_JSON_RE = None  # (regex-এর বদলে নিচে plain check function রাখা হয়েছে, স্পষ্টতার জন্য)


def is_fully_valid(s):
    """Independent ground-truth validity check, নিচের grammar state machine
    থেকে সম্পূর্ণ আলাদাভাবে লেখা -- যাতে সে সত্যিই `next_valid_chars`-এর
    bug-টা ধরতে পারে, construction-এর কারণেই ওর সাথে একমত হয়ে যাওয়ার
    বদলে।"""
    if not s.startswith(TEMPLATE_HEAD):
        return False
    rest = s[len(TEMPLATE_HEAD):]
    for action in ACTIONS:
        prefix = action + TEMPLATE_MID
        if rest.startswith(prefix):
            tail = rest[len(prefix):]
            if tail.endswith("}"):
                digits = tail[:-1]
                if digits.isdigit() and (digits == "0" or not digits.startswith("0")) and 1 <= len(digits) <= 3:
                    return True
    return False


def next_valid_chars(prefix):
    """Grammar-টা, একটা character-level state machine হিসেবে implement করা
    যেটা purely `prefix` inspect করে চলে। বৈধ পরের characters-এর set return
    করে, অথবা খালি set মানে 'structure সম্পূর্ণ -- থামো।'"""
    if len(prefix) < len(TEMPLATE_HEAD):
        return {TEMPLATE_HEAD[len(prefix)]}
    rest = prefix[len(TEMPLATE_HEAD):]

    quote_idx = rest.find('"')
    if quote_idx == -1:
        partial = rest  # mid-way through the enum word, no closing quote yet
        valid = {w[len(partial)] for w in ACTIONS if w.startswith(partial) and len(w) > len(partial)}
        if partial in ACTIONS:
            valid.add('"')          # the word is complete -- allowed to close the string now
        return valid

    partial = rest[:quote_idx]      # guaranteed to be a full action word (see loop invariant below)
    after_quote = rest[quote_idx + 1:]
    if len(after_quote) < len(TEMPLATE_MID):
        return {TEMPLATE_MID[len(after_quote)]}

    digits_and_after = after_quote[len(TEMPLATE_MID):]
    brace_idx = digits_and_after.find("}")
    if brace_idx != -1:
        return set()               # closing brace already emitted -- structure complete, stop

    digits_so_far = digits_and_after
    if digits_so_far == "":
        return set("0123456789")                 # first digit: anything, including a lone '0'
    if digits_so_far == "0":
        return {"}"}                              # "0" cannot be followed by more digits (no leading zeros)
    if len(digits_so_far) < 3:
        return set("0123456789") | {"}"}          # 1-2 digits so far: may extend or close
    return {"}"}                                  # 3 digits: MUST close now


# --- একটা ছোট্ট, সত্যিই UNTRAINED decoder-only Transformer (random weights) ---

VOCAB_CHARS = sorted(set(TEMPLATE_HEAD + TEMPLATE_MID + "".join(ACTIONS) + "0123456789}"))
vocab_size = len(VOCAB_CHARS)
stoi = {ch: i for i, ch in enumerate(VOCAB_CHARS)}
itos = {i: ch for i, ch in enumerate(VOCAB_CHARS)}
BLOCK_SIZE = 40


class TinyCausalTransformer(nn.Module):
    """একটা ছোট্ট causal self-attention block। weights-গুলো উদ্দেশ্য নিয়ে
    তাদের random initialization-এ রেখে দেওয়া হয় -- Part A-র পুরো বক্তব্য
    হলো constrained decoding syntactic validity guarantee করে, underlying
    model কিছু শিখেছে কি না তা নির্বিশেষে।"""

    def __init__(self, vocab_size, d_model=32, num_heads=2, block_size=BLOCK_SIZE):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.num_heads = num_heads
        self.d_model = d_model
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).bool())

    def forward(self, ids):
        batch, T = ids.shape
        positions = torch.arange(T)
        x = self.token_embedding(ids) + self.position_embedding(positions)
        h = self.ln1(x)
        d_k = self.d_model // self.num_heads
        Q = self.W_q(h).view(batch, T, self.num_heads, d_k).transpose(1, 2)
        K = self.W_k(h).view(batch, T, self.num_heads, d_k).transpose(1, 2)
        V = self.W_v(h).view(batch, T, self.num_heads, d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)
        scores = scores.masked_fill(~self.mask[:T, :T], float("-inf"))
        attn = (F.softmax(scores, dim=-1) @ V).transpose(1, 2).reshape(batch, T, self.d_model)
        x = x + attn
        x = x + self.ffn(self.ln2(x))
        return self.head(x)


@torch.no_grad()
def generate(model, constrained, max_len=BLOCK_SIZE, temperature=1.0, rng=None):
    """Character-by-character একটা sequence generate করো। `constrained` True
    হলে grammar-ভঙ্গকারী characters-এর logits-গুলো প্রতিটি sampling step-এর
    আগে -inf-এ set হয়। False হলে model পুরো vocabulary-র ওপর free-ভাবে
    sample করে -- যে baseline-এর সাথে আমরা তুলনা করি।"""
    prefix = ""
    for _ in range(max_len):
        if constrained:
            valid = next_valid_chars(prefix)
            if not valid:
                break   # grammar বলে: structure সম্পূর্ণ
        ids = torch.tensor([[stoi[c] for c in prefix]], dtype=torch.long) if prefix else torch.zeros((1, 1), dtype=torch.long)
        if prefix == "":
            logits = model(torch.zeros((1, 1), dtype=torch.long))[0, -1]
            # (খালি prefix-এ এখনো কোনো token নেই; প্রথম step-এর logit vector
            # পেতে আমাদের একটা forward pass দরকার, তাই dummy token 0 দিয়ে
            # seed করি আর নিচের masking দিয়ে সাথে সাথে overwrite/ignore করি)
        else:
            logits = model(ids)[0, -1]
        if constrained:
            mask = torch.full((vocab_size,), float("-inf"))
            for c in valid:
                mask[stoi[c]] = 0.0
            logits = logits + mask
        probs = F.softmax(logits / temperature, dim=-1)
        if torch.isnan(probs).any():
            break
        next_id = torch.multinomial(probs, num_samples=1, generator=rng).item()
        prefix += itos[next_id]
        if not constrained and len(prefix) >= max_len:
            break
    return prefix


def part_a_demo():
    print("=" * 78)
    print("PART A: CONSTRAINED DECODING GUARANTEES VALID STRUCTURED OUTPUT")
    print("=" * 78)
    print(f"Grammar: {TEMPLATE_HEAD}<action>{TEMPLATE_MID}<1-3 digit value>}}")
    print(f"  <action> in {ACTIONS}")
    print(f"Vocabulary ({vocab_size} characters): {VOCAB_CHARS}")

    model = TinyCausalTransformer(vocab_size)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel: a {num_params:,}-parameter causal Transformer -- weights are")
    print("PURE RANDOM INITIALIZATION. No training happens anywhere in Part A.")

    print("\n" + "-" * 78)
    print("Self-check: verifying the grammar state machine against an INDEPENDENT")
    print("full-string validity checker, using pure random valid-choice walks")
    print("-" * 78)
    self_check_trials = 2000
    self_check_failures = 0
    for _ in range(self_check_trials):
        prefix = ""
        for _ in range(BLOCK_SIZE):
            valid = next_valid_chars(prefix)
            if not valid:
                break
            prefix += random.choice(sorted(valid))
        if not is_fully_valid(prefix):
            self_check_failures += 1
    print(f"{self_check_trials} random walks through the grammar's own valid-character sets;")
    print(f"failures against the INDEPENDENT validity checker: {self_check_failures}")
    print(f"-> The state machine and the independent checker agree {self_check_failures == 0}: "
          f"every path the grammar permits is genuinely valid.")

    print("\n" + "-" * 78)
    print("CONSTRAINED generations from the UNTRAINED model (mask applied every step)")
    print("-" * 78)
    num_generations = 8
    constrained_outputs = []
    for i in range(num_generations):
        rng = torch.Generator().manual_seed(100 + i)
        out = generate(model, constrained=True, rng=rng)
        constrained_outputs.append(out)
        print(f"  [{i}] {out!r}   valid={is_fully_valid(out)}")
    num_valid_constrained = sum(is_fully_valid(o) for o in constrained_outputs)
    print(f"\nValid outputs: {num_valid_constrained}/{num_generations}")

    print("\n" + "-" * 78)
    print("UNCONSTRAINED generations from the SAME untrained model (no mask at all)")
    print("-" * 78)
    unconstrained_outputs = []
    for i in range(num_generations):
        rng = torch.Generator().manual_seed(100 + i)
        out = generate(model, constrained=False, max_len=24, rng=rng)
        unconstrained_outputs.append(out)
        print(f"  [{i}] {out!r}   valid={is_fully_valid(out)}")
    num_valid_unconstrained = sum(is_fully_valid(o) for o in unconstrained_outputs)
    print(f"\nValid outputs: {num_valid_unconstrained}/{num_generations}")

    print(f"\n-> Same random, untrained model, same random seeds, same sampling procedure.")
    print(f"   With the grammar mask applied at every step: {num_valid_constrained}/{num_generations} valid.")
    print(f"   With no mask at all:                          {num_valid_unconstrained}/{num_generations} valid.")
    print("   Validity here comes ENTIRELY from masking illegal tokens to -inf before")
    print("   sampling, not from anything the model has learned -- constrained decoding")
    print("   makes a syntax guarantee that holds regardless of model quality, which is")
    print("   exactly why production structured-output APIs implement it at the")
    print("   decoding layer instead of just hoping a well-trained model complies.")
part_a_demo()

## Part B — Function calling: request → parse → execute → respond

Tool implementations (calculator, unit-converter), `TOOLS` register, আর `run_function_call` — যা scripted model output থেকে JSON request parse করে আসল function-টা run করে। Cell-এর শেষে `part_b_demo()` দুইটা scenario চালিয়ে verification-ও করে।

In [ ]:
# ===========================================================================
# PART B: FUNCTION CALLING -- REQUEST, PARSE, EXECUTE, RESPOND
# ===========================================================================

def tool_calculator(expression):
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError(f"Unsupported characters in expression: {expression!r}")
    return eval(expression, {"__builtins__": {}}, {})


UNIT_CONVERSIONS = {
    ("miles", "km"): 1.60934,
    ("km", "miles"): 1 / 1.60934,
    ("kg", "lb"): 2.20462,
    ("lb", "kg"): 1 / 2.20462,
}


def tool_convert_units(value, from_unit, to_unit):
    factor = UNIT_CONVERSIONS.get((from_unit, to_unit))
    if factor is None:
        raise ValueError(f"No conversion known for {from_unit} -> {to_unit}")
    return value * factor


TOOLS = {
    "calculator": tool_calculator,
    "convert_units": tool_convert_units,
}


def run_function_call(model_output_text):
    """(এখানে: scripted) model output string-এর ভেতর থেকে একটা structured
    function-call request parse করে, REAL parsed arguments দিয়ে প্রকৃত
    সংশ্লিষ্ট Python function execute করে, আর tool-এর real result return করে।"""
    call = json.loads(model_output_text)
    name = call["name"]
    arguments = call["arguments"]
    if name not in TOOLS:
        raise ValueError(f"Model requested unknown tool: {name!r}")
    result = TOOLS[name](**arguments)
    return name, arguments, result


def part_b_demo():
    print("\n" + "=" * 78)
    print("PART B: FUNCTION CALLING -- STRUCTURED REQUEST -> PARSE -> EXECUTE -> RESPOND")
    print("=" * 78)
    print("Each 'model output' below is a SCRIPTED string (there is no LLM generating")
    print("it) standing in for what a real model would emit in the standard")
    print('{"name": ..., "arguments": {...}} function-calling format. Everything AFTER')
    print("that point -- JSON parsing, tool dispatch, and execution -- is real code")
    print("running on real inputs, with a real return value spliced back in.\n")

    scenarios = [
        {
            "user_query": "What is 128 times 37, plus 6?",
            "model_output": '{"name": "calculator", "arguments": {"expression": "128 * 37 + 6"}}',
            "response_template": "The result of 128 * 37 + 6 is {result}.",
        },
        {
            "user_query": "Convert 42 kilometers to miles.",
            "model_output": '{"name": "convert_units", "arguments": {"value": 42, "from_unit": "km", "to_unit": "miles"}}',
            "response_template": "42 km is approximately {result:.2f} miles.",
        },
    ]

    for i, scenario in enumerate(scenarios, start=1):
        print(f"--- Scenario {i} ---")
        print(f"User query:    {scenario['user_query']}")
        print(f"Model output:  {scenario['model_output']}")
        name, arguments, result = run_function_call(scenario["model_output"])
        print(f"Parsed call:   name={name!r}, arguments={arguments}")
        print(f"Tool executed. Real return value: {result!r}")
        final_response = scenario["response_template"].format(result=result)
        print(f"Final response (real tool result spliced in): {final_response!r}\n")

    # Splice-করা সংখ্যাগুলো সত্যিই সঠিক কিনা তার independent verification,
    # উপরের tool-calling machinery থেকে সম্পূর্ণ আলাদাভাবে compute করা।
    expected_1 = 128 * 37 + 6
    expected_2 = 42 * (1 / 1.60934)
    _, _, result_1 = run_function_call(scenarios[0]["model_output"])
    _, _, result_2 = run_function_call(scenarios[1]["model_output"])
    print(f"Independent check -- scenario 1: expected {expected_1}, tool returned {result_1}, "
          f"match={expected_1 == result_1}")
    print(f"Independent check -- scenario 2: expected {expected_2:.4f}, tool returned {result_2:.4f}, "
          f"match={abs(expected_2 - result_2) < 1e-9}")
    print("\n-> Both final responses embed a number the calling code could not have")
    print("   known in advance without actually running the requested tool -- this is")
    print("   the entire value of function calling: the model's job is reduced to")
    print("   emitting a syntactically valid REQUEST (which Part A showed can be")
    print("   guaranteed structurally), while a real system executes it and supplies")
    print("   the real answer back into the conversation.")
part_b_demo()

## পুরো demonstration চালানো

`main()` function-টা `part_a_demo()` আর `part_b_demo()` দুটোই call করে; শেষ cell-এ `main()` কল হয়। (Section cells-এ demo-গুলো ইতিমধ্যে run হয়ে গেছে, তাই `main()` চালালে আউটপুট আবার print হবে — প্রত্যাশিত।)

In [ ]:
def main():
    part_a_demo()
    part_b_demo()

In [ ]:
main()